# تقييم جودة بيانات العملاء
شغّل الخلايا بالترتيب. ينتج المحرك ملف JSON يضم نتائج القواعد لكل حقل، ثم تعرضه اللوحة. تاريخ القياس الثابت هو تاريخ لقطة التدريب 2026-11-18. تُحسب نسب المطابقة بين السجلات المؤهلة لكل قاعدة. يعرض التقرير عدد المستبعدين وأرقام الصفوف المخالفة (بدءًا من أول سجل بيانات).

In [ ]:
"""Evaluate the course customer schema. Run from project root: python scripts/assess_customers.py"""
import argparse
import csv
import json
import re
from collections import Counter
from datetime import date
from pathlib import Path

FIELDS = "customer_id full_name email mobile_number national_id city date_of_birth registration_date last_updated_at customer_status".split()
DIMS = ["الاكتمال", "الصلاحية", "التفرّد", "الاتساق", "الدقة", "الحداثة"]
SNAPSHOT = date(2026, 11, 18)
CITIES = {"الرياض", "جدة", "الدمام", "مكة المكرمة", "المدينة المنورة", "أبها", "تبوك"}
STATUSES = {"نشط", "غير نشط", "معلق"}
UNIQUE = {"customer_id", "email", "national_id", "mobile_number"}
PATTERNS = {"customer_id": r"CUST-[0-9]{5}", "email": r"[^\s@]+@[^\s@]+\.[^\s@]+",
            "mobile_number": r"05[0-9]{8}", "national_id": r"[12][0-9]{9}"}


def read_rows(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        if not set(FIELDS).issubset(reader.fieldnames or []):
            raise ValueError(f"أعمدة مطلوبة مفقودة في {path}")
        return [{f: (row[f] or "").strip() for f in FIELDS} for row in reader]


def as_date(value):
    if not re.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}", value):
        return None
    try:
        return date.fromisoformat(value)
    except ValueError:
        return None


def validity(field, value):
    if field in PATTERNS:
        return bool(re.fullmatch(PATTERNS[field], value))
    if field == "city":
        return value in CITIES
    if field == "customer_status":
        return value in STATUSES
    if field == "full_name":
        return len(value.split()) >= 2 and all(c.isalpha() or c.isspace() for c in value)
    parsed = as_date(value)
    if field == "registration_date":
        return parsed is not None and parsed >= date(2022, 1, 1)
    return parsed is not None


def assess(rows, reference=None, snapshot=SNAPSHOT):
    counts = {f: Counter(r[f] for r in rows if r[f]) for f in UNIQUE}
    refcounts = Counter(r["customer_id"] for r in reference or [])
    refs = {r["customer_id"]: r for r in reference or [] if r["customer_id"] and refcounts[r["customer_id"]] == 1}
    cells = []

    def record(field, dim, rule, check=None, unavailable=False):
        cell = dict(field=field, dimension=dim, rule=rule, passed=0, evaluated=0,
                    excluded=len(rows), percent=None, issues=[], status="ينطبق")
        if check is None:
            cell["status"] = "غير قابل للقياس" if unavailable else "لا ينطبق"
        else:
            for number, row in enumerate(rows, 1):
                result = check(row)
                if result is None:
                    continue
                cell["evaluated"] += 1
                if result:
                    cell["passed"] += 1
                else:
                    cell["issues"].append({"row": number, "customer_id": row["customer_id"]})
            cell["excluded"] -= cell["evaluated"]
            if cell["evaluated"]:
                cell["percent"] = round(100 * cell["passed"] / cell["evaluated"], 2)
            else:
                cell["status"] = "غير قابل للقياس"
        cells.append(cell)

    valid_desc = {**PATTERNS, "city": "مدينة من قائمة المدن التدريبية",
                  "customer_status": "نشط أو غير نشط أو معلق",
                  "full_name": "كلمتان على الأقل من الحروف والمسافات (قاعدة تدريبية)",
                  "registration_date": "تاريخ ISO صالح ابتداءً من 2022-01-01"}
    pairs = {"date_of_birth": ("registration_date",), "registration_date": ("date_of_birth", "last_updated_at"),
             "last_updated_at": ("registration_date",)}
    for f in FIELDS:
        record(f, DIMS[0], "وجود قيمة بعد حذف المسافات الطرفية", lambda r: bool(r[f]))
        record(f, DIMS[1], valid_desc.get(f, "تاريخ صالح بصيغة YYYY-MM-DD"),
               lambda r: validity(f, r[f]) if r[f] else None)
        if f in UNIQUE:
            record(f, DIMS[2], "القيمة موجودة مرة واحدة بين القيم المعبأة؛ يُعلَّم كل طرف من التكرار",
                   lambda r: counts[f][r[f]] == 1 if r[f] else None)
        else:
            record(f, DIMS[2], "التكرار مقبول لهذا الحقل")
        if f in pairs:
            def consistent(r):
                needed = [f, *pairs[f]]
                dates = {k: as_date(r[k]) for k in needed}
                if any(v is None for v in dates.values()):
                    return None
                birth, reg, update = (dates.get(k) for k in ("date_of_birth", "registration_date", "last_updated_at"))
                return (birth is None or birth < reg) and (update is None or reg <= update)
            descriptions = {"date_of_birth": "الميلاد يسبق التسجيل", "registration_date": "الميلاد يسبق التسجيل والتسجيل لا يتجاوز آخر تحديث", "last_updated_at": "آخر تحديث يساوي تاريخ التسجيل أو يليه"}
            record(f, DIMS[3], descriptions[f], consistent)
        else:
            record(f, DIMS[3], "لم تُعرّف علاقة بين هذا الحقل وحقل آخر في خطة المشروع")
        if f == "customer_id":
            record(f, DIMS[4], "مفتاح الربط نفسه يحتاج إلى مرجع مستقل للتحقق من دقته", unavailable=True)
        elif reference is None:
            record(f, DIMS[4], "الملف المرجعي غير متوفر", unavailable=True)
        else:
            def accurate(r):
                key = r["customer_id"]
                ref = refs.get(key)
                if not ref or counts["customer_id"][key] != 1 or not r[f] or not ref[f]:
                    return None
                return r[f] == ref[f]
            record(f, DIMS[4], "مطابقة قيمة معبأة مع المرجع التدريبي باستخدام معرّف عميل فريد في الملفين", accurate)
        if f == "last_updated_at":
            def recent(r):
                updated = as_date(r[f])
                return None if updated is None else 0 <= (snapshot - updated).days <= 90
            record(f, DIMS[5], f"عمر آخر تحديث بين 0 و90 يومًا عند {snapshot.isoformat()}", recent)
        else:
            record(f, DIMS[5], "تُقاس حداثة السجل من تاريخ آخر تحديث")
    return {"rows": len(rows), "snapshot": snapshot.isoformat(), "dimensions": DIMS, "fields": FIELDS, "cells": cells}


def save_report(report, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

In [ ]:
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve

def measure_course_data():
    names = ["customers_raw.csv", "customers_clean_reference.csv"]
    for folder in [Path("data"), Path("../data"), Path("/content/customer-data-quality-project/data"), Path("/content/data-quality-inspector/data")]:
        if all((folder / name).is_file() for name in names):
            print("مصدر البيانات:", folder.resolve())
            return assess(*(read_rows(folder / name) for name in names))
    base = "https://raw.githubusercontent.com/mutebalotaibi/data-quality-inspector/main/data/"
    with TemporaryDirectory() as tmp:
        print("مصدر البيانات:", base)
        paths = [Path(tmp) / name for name in names]
        for path in paths:
            urlretrieve(base + path.name, path)
        return assess(*(read_rows(path) for path in paths))

report_path = Path("reports/quality_results.json")
quality = measure_course_data()
save_report(quality, report_path)
print("عدد السجلات:", quality["rows"])
print("عدد خلايا المصفوفة:", len(quality["cells"]))
print("ملف النتائج:", report_path.resolve())

In [ ]:
import pandas as pd

results = pd.DataFrame(quality["cells"])
results["النتيجة"] = [c["status"] if c["percent"] is None else f'{c["percent"]:.2f}%' for c in quality["cells"]]
display(results.pivot(index="field", columns="dimension", values="النتيجة").reindex(index=FIELDS, columns=DIMS))

## لوحة المعلومات
تعرض الخلية التالية تقرير القياس المحفوظ. اضغط أي خلية في المصفوفة لعرض القاعدة والتفاصيل. تجد نسخة HTML مستقلة داخل مجلد `reports` في ملفات الجلسة.

In [ ]:
"""Render the assessment JSON as a self-contained Arabic HTML dashboard."""
from html import escape
from pathlib import Path
import json
import sys


PAGE = """<!doctype html><html lang="ar" dir="rtl"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>لوحة جودة بيانات العملاء</title>
<style>
*{box-sizing:border-box}body{margin:0;background:#f5f6f2;color:#183a38;font:16px/1.65 Tahoma,Arial,sans-serif}
main{max-width:1280px;margin:auto;padding:28px clamp(12px,4vw,54px)}header{border-bottom:3px solid #1b8177;padding-bottom:18px}
h1,h2,p{margin:.25em 0}h1{font-size:clamp(24px,4vw,40px)}h2{font-size:21px}section{margin-top:24px}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(155px,1fr));gap:12px}.card,.panel{background:white;border:1px solid #d1dcd5;border-radius:12px;padding:16px}
.card strong{display:block;font:700 30px/1.4 Arial;color:#12665f}.scroll{overflow:auto}table{width:100%;border-collapse:collapse;white-space:nowrap}
td,th{padding:8px;text-align:center;border-bottom:1px solid #e3e9e5}th[scope=row]{text-align:start}
thead{background:#eff4ef}button,select{font:inherit}td button{border:1px solid transparent;border-radius:6px;padding:5px 8px;width:100%;cursor:pointer;color:#174d46;background:#dff1e6}
td button.problem{background:#fff0d3;color:#734900}td button.na{background:#f1f2f3;color:#505e60;font-size:13px}
td button:focus-visible,td button.active{outline:2px solid #185d73;outline-offset:1px}label{display:inline-block;margin:8px 0 8px 18px}select{padding:4px}
code,bdi{direction:ltr;unicode-bidi:isolate}#issues{max-height:200px;overflow:auto}#issues li{display:inline-block;margin:3px;padding:4px 8px;background:#fff3de;border-radius:4px}#issues{padding:0}
</style><main><header><h1>جودة بيانات العملاء</h1><p>تقرير البيانات التجريبية · تاريخ القياس <bdi id="day"></bdi></p></header>
<section class="kpis" aria-label="ملخص"><div class="card">سجلات العملاء<strong id="rows"></strong></div><div class="card">اكتمال الخانات الإلزامية<strong id="complete"></strong></div><div class="card">سجلات لها مخالفة واحدة على الأقل<strong id="affected"></strong></div></section>
<section class="panel"><h2>مصفوفة جودة الحقول</h2><p>اختر نسبة لعرض القاعدة والسجلات المخالفة. النسبة = عدد القيم المطابقة ÷ عدد القيم المفحوصة.</p>
<div class="scroll"><table><thead id="head"></thead><tbody id="matrix"></tbody></table></div></section>
<section id="detail" class="panel" tabindex="-1"><h2>تفاصيل القياس</h2><label>الحقل <select id="field" dir="ltr"></select></label><label>البُعد <select id="dim"></select></label>
<div aria-live="polite"><p id="rule"></p><p id="stat"></p><ul id="issues"></ul></div></section></main>
<script type="application/json" id="payload">__REPORT__</script><script>
const data=JSON.parse(document.getElementById('payload').textContent);
const el=id=>document.getElementById(id);
function add(tag,parent,text){const node=document.createElement(tag);parent.appendChild(node);if(text!==undefined)node.textContent=text;return node}
const complete=data.cells.filter(c=>c.dimension===data.dimensions[0]);
const total=complete.reduce((n,c)=>n+c.evaluated,0), filled=complete.reduce((n,c)=>n+c.passed,0);
el('rows').textContent=data.rows;el('day').textContent=data.snapshot;el('complete').textContent=total?(100*filled/total).toFixed(2)+'%':'غير قابل للقياس';
el('affected').textContent=new Set(data.cells.flatMap(c=>c.issues.map(i=>i.row))).size;
for(const f of data.fields) el('field').add(new Option(f,f));
for(const d of data.dimensions) el('dim').add(new Option(d,d));
const header=add('tr',el('head'));for(const name of ['الحقل',...data.dimensions])add('th',header,name).scope='col';
function show(){
 const f=el('field').value,d=el('dim').value,c=data.cells.find(x=>x.field===f&&x.dimension===d);
 el('rule').textContent=c.rule;
 el('stat').textContent=c.percent===null?c.status:'المطابق '+c.passed+' من '+c.evaluated+' مفحوصًا · المخالف '+c.issues.length+' · المستبعد من المقام '+c.excluded;
 el('issues').replaceChildren();
 for(const i of c.issues)add('li',el('issues'),'السجل '+i.row+' · '+i.customer_id);
 for(const b of el('matrix').querySelectorAll('button'))b.classList.toggle('active',b.dataset.f===f&&b.dataset.d===d);
}
for(const f of data.fields){
 const tr=add('tr',el('matrix'));const title=add('th',tr,f);title.scope='row';title.dir='ltr';
 for(const d of data.dimensions){
 const c=data.cells.find(x=>x.field===f&&x.dimension===d),td=add('td',tr),label=c.percent===null?c.status:c.percent.toFixed(2)+'%';
 const b=add('button',td,label);b.dataset.f=f;b.dataset.d=d;b.className=c.percent===null?'na':c.issues.length?'problem':'';
 b.setAttribute('aria-label',f+'، '+d+'، '+label);
 b.onclick=()=>{el('field').value=f;el('dim').value=d;show();el('detail').focus()};}}
el('field').onchange=el('dim').onchange=show;el('field').value='email';el('dim').value=data.dimensions[1];show();
</script></html>"""


def render_dashboard(report_path, output_path):
    report = json.loads(Path(report_path).read_text(encoding="utf-8"))
    payload = json.dumps(report, ensure_ascii=False).replace("<", "\\u003c")
    Path(output_path).write_text(PAGE.replace("__REPORT__", payload), encoding="utf-8")

In [ ]:
from IPython.display import HTML, display
from html import escape

dashboard_path = report_path.with_name("dashboard.html")
render_dashboard(report_path, dashboard_path)
content = escape(dashboard_path.read_text(encoding="utf-8"), quote=True)
display(HTML(f'<iframe title="جودة بيانات العملاء" sandbox="allow-scripts" style="width:100%;height:950px;border:0" srcdoc="{content}"></iframe>'))
print("ملف اللوحة:", dashboard_path.resolve())